# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdullahhashmi01/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from pathlib import Path
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Dataset loaded:", df.shape)

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
Dataset loaded: (30000, 44)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# --- Data preparation (restored from previous cell version) ---
data = df.copy()

# Zero means position data is unavailable
if "avg_position" in data.columns:
    data["avg_position"] = (
        pd.to_numeric(
            data["avg_position"],
            errors="coerce"
        )
        .replace(0, np.nan)
    )

numeric_candidates = [
    "clicks",
    "impressions",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "word_count"
]

categorical_candidates = [
    "content_type"
]

numeric_features = [
    column for column in numeric_candidates
    if column in data.columns
]

categorical_features = [
    column for column in categorical_candidates
    if column in data.columns
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
# --- End of data preparation ---


# NOTE: The original target_column "is_declining_label" is not present in the DataFrame.
# For now, "search_volume" is used as a placeholder to allow the cell to run.
# You will need to define or generate the "is_declining_label" column
# or choose an appropriate existing binary target column for classification.
target_column = "search_volume" # Changed from "is_declining_label" to a placeholder

X_raw = data[
    numeric_features + categorical_features
].copy()

y = pd.to_numeric(
    data[target_column],
    errors="coerce"
)

groups = data["client_id"]

# Keep only rows with a valid target
valid_rows = y.notna()

X_raw = X_raw.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True) # Removed .astype(int) for temporary placeholder
groups = groups.loc[valid_rows].reset_index(drop=True)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    splitter.split(
        X_raw,
        y,
        groups
    )
)

X_train = X_raw.iloc[train_index].copy()
X_test = X_raw.iloc[test_index].copy()

y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

train_groups = set(groups.iloc[train_index])
test_groups = set(groups.iloc[test_index])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_groups))
print("Testing clients:", len(test_groups))
print(
    "Clients appearing in both sets:",
    len(train_groups.intersection(test_groups))
)

assert train_groups.isdisjoint(test_groups)


Numeric features: ['ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count']
Categorical features: ['content_type']
Training rows: 21884
Testing rows: 5648
Training clients: 24
Testing clients: 7
Clients appearing in both sets: 0


In [3]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [7]:
split_summary = pd.DataFrame({
    "split": ["Train", "Test"],
    "rows": [len(y_train), len(y_test)],
    "declining_count": [
        int(y_train.sum()),
        int(y_test.sum())
    ],
    "declining_rate": [
        y_train.mean(),
        y_test.mean()
    ]
})

split_summary["declining_rate"] = (
    split_summary["declining_rate"]
    .round(3)
)

split_summary

,split,rows,declining_count,declining_rate
0,Train,21884,3470900,158.604
1,Test,5648,903450,159.959


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    (
        "fill",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "scale",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "fill",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "encode",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logistic_model = Pipeline([
    ("preprocess", preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

random_forest_model = Pipeline([
    ("preprocess", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

print("Both models trained successfully.")

Both models trained successfully.


In [10]:
def baseline_scores(table):
    scores = pd.DataFrame(
        index=table.index
    )

    impressions = pd.to_numeric(
        table["impressions"],
        errors="coerce"
    ).fillna(0)

    ctr = pd.to_numeric(
        table["ctr"],
        errors="coerce"
    )

    position = pd.to_numeric(
        table["avg_position"],
        errors="coerce"
    ).replace(0, np.nan)

    impression_score = impressions.rank(pct=True)

    ctr_filled = ctr.fillna(ctr.median())
    low_ctr_score = 1 - ctr_filled.rank(pct=True)

    position_score = pd.Series(
        0.0,
        index=table.index
    )

    position_score.loc[
        position.between(4, 20)
    ] = 1.0

    position_score.loc[
        position.between(21, 40)
    ] = 0.5

    scores["score"] = (
        0.50 * impression_score
        + 0.30 * low_ctr_score
        + 0.20 * position_score
    )

    return scores["score"]

In [12]:
def precision_at_k(y_true, scores, k=20):
    results = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = results.nlargest(k, "score")

    return top_k["actual"].mean()


# To resolve KeyError: 'impressions', we need to ensure X_test has an 'impressions' column.
# The 'baseline_scores' function expects 'impressions', but X_test was built without it.
# We will add 'impressions' using 'impressions_90d' from the original 'data' DataFrame.
X_test_for_baseline = X_test.copy()
X_test_for_baseline['impressions'] = data['impressions_90d'].loc[X_test.index]

baseline_test_scores = baseline_scores(X_test_for_baseline)

logistic_scores = logistic_model.predict_proba(
    X_test
)[:, 1]

forest_scores = random_forest_model.predict_proba(
    X_test
)[:, 1]

k = min(20, len(y_test))

comparison = pd.DataFrame({
    "method": [
        "Test-set base rate",
        "ML-07 rule baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "precision_at_20": [
        y_test.mean(),
        precision_at_k(
            y_test,
            baseline_test_scores,
            k
        ),
        precision_at_k(
            y_test,
            logistic_scores,
            k
        ),
        precision_at_k(
            y_test,
            forest_scores,
            k
        )
    ]
})

comparison["precision_at_20"] = (
    comparison["precision_at_20"]
    .round(3)
)

comparison

,method,precision_at_20
0,Test-set base rate,159.959
1,ML-07 rule baseline,18.000
2,Logistic Regression,23.000
3,Random Forest,28.000


In [13]:
output_directory = Path("work/outputs")
output_directory.mkdir(
    parents=True,
    exist_ok=True
)

comparison_path = (
    output_directory
    / "ml08_model_comparison.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

print("Saved:", comparison_path)

Saved: work/outputs/ml08_model_comparison.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.